# Latent Semantic Analysis (LSA)

$$
A = U \Sigma V^\top
$$

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go

from nltk.corpus import reuters
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score

nltk.download('reuters', quiet=True)

## Data

In [ ]:
categories = ["gas", "cpi", "coffee"]

fileids_by_topic = {topic: set(reuters.fileids([topic])) for topic in categories}

texts = []
topics = []
for topic in categories:
    # keep only the articles that belong to exactly one of the three categories
    single_label = sorted(
        fileid for fileid in fileids_by_topic[topic]
        if sum(fileid in fileids for fileids in fileids_by_topic.values()) == 1
    )
    texts += [reuters.raw(fileid) for fileid in single_label]
    topics += [topic] * len(single_label)

topics = np.array(topics)
print(pd.Series(topics).value_counts())

## The TF-IDF Matrix

In [ ]:
texts = [re.sub(r"\d+", "", text) for text in texts]

tfidf = TfidfVectorizer(lowercase=True, stop_words="english", min_df=0.05)
term_doc_matrix = tfidf.fit_transform(texts).toarray()   # n documents x m terms, rows have unit length
terms = np.array(tfidf.get_feature_names_out())

print("Shape (documents x terms):", term_doc_matrix.shape)
print("Most important terms (column sums):")
print(pd.Series(term_doc_matrix.sum(axis=0), index=terms).sort_values(ascending=False).head(10).round(1))

## Helper Functions

In [ ]:
def lsa_svd(matrix):
    """SVD of a matrix: document scores (U * Sigma), singular values and the term loadings (rows of Vt)."""
    U, sigma, Vt = np.linalg.svd(matrix, full_matrices=False)
    # fix the arbitrary sign: the largest absolute loading of every component is positive
    signs = np.sign(Vt[np.arange(len(sigma)), np.abs(Vt).argmax(axis=1)])
    return {"scores": (U * sigma) * signs, "sigma": sigma, "Vt": Vt * signs[:, None],
            "share": sigma**2 / np.sum(sigma**2)}


def plot_scree(result, title, n_show=15):
    share = result["share"][:n_show]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(np.arange(1, n_show + 1), share)
    ax.plot(np.arange(1, n_show + 1), np.cumsum(share), "o-", color="black", label="cumulative")
    ax.set_xlabel("Component")
    ax.set_ylabel("Share of the sum of squares")
    ax.set_title(title)
    ax.legend()
    plt.show()


def top_terms(result, n_terms=8, n_components=3):
    """Terms with the most positive and the most negative loadings of the first components."""
    table = {}
    for k in range(n_components):
        order = np.argsort(result["Vt"][k])
        table[f"PC{k + 1} (+)"] = [f"{terms[i]} ({result['Vt'][k, i]:.2f})" for i in order[::-1][:n_terms]]
        table[f"PC{k + 1} (-)"] = [f"{terms[i]} ({result['Vt'][k, i]:.2f})" for i in order[:n_terms]]
    return pd.DataFrame(table)


topic_colors = {"gas": "#EF553B", "cpi": "#636EFA", "coffee": "#00CC96"}


def plot_documents_3d(result, title):
    scores, share = result["scores"], result["share"]
    fig = go.Figure()
    for topic in categories:
        mask = topics == topic
        fig.add_trace(go.Scatter3d(
            x=scores[mask, 0], y=scores[mask, 1], z=scores[mask, 2], mode="markers", name=topic,
            marker=dict(size=4, color=topic_colors[topic], opacity=0.8, line=dict(width=0.5, color="black"))))
    fig.update_layout(
        title=title, width=800, height=700,
        scene=dict(xaxis_title=f"PC1 ({share[0]:.0%})", yaxis_title=f"PC2 ({share[1]:.0%})",
                   zaxis_title=f"PC3 ({share[2]:.0%})"),
        legend=dict(itemsizing="constant"))
    fig.show()


def plot_documents_2d(result, title):
    scores, share = result["scores"], result["share"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (i, j) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
        for topic in categories:
            mask = topics == topic
            ax.scatter(scores[mask, i], scores[mask, j], s=12, alpha=0.7, color=topic_colors[topic], label=topic)
        ax.set_xlabel(f"PC{i + 1} ({share[i]:.0%})")
        ax.set_ylabel(f"PC{j + 1} ({share[j]:.0%})")
    axes[0].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Analysis A: SVD of the Raw TF-IDF Matrix

In [ ]:
raw = lsa_svd(term_doc_matrix)

# TruncatedSVD gives the same singular values
truncated = TruncatedSVD(n_components=3, n_iter=100, random_state=42).fit(term_doc_matrix)
print("TruncatedSVD equals the SVD of A:", np.allclose(truncated.singular_values_, raw["sigma"][:3]))

mean_document = term_doc_matrix.mean(axis=0)
cosine_with_mean = raw["Vt"][0] @ mean_document / np.linalg.norm(mean_document)
print("All loadings of PC1 positive:", bool((raw["Vt"][0] > 0).all()))
print("Cosine between the first right singular vector and the mean document:", round(float(cosine_with_mean), 3))
print("Share of the sum of squares of the first three components:", raw["share"][:3].round(3))

In [ ]:
plot_scree(raw, "Raw TF-IDF matrix")

In [ ]:
top_terms(raw)

In [ ]:
plot_documents_3d(raw, "Documents in the space of the first three components (raw)")

### Interpretation (raw)

## Analysis B: SVD of the Centered TF-IDF Matrix

$$
A - \mathbf{1} \bar{a}^\top = U \Sigma V^\top
$$

In [ ]:
centered_matrix = term_doc_matrix - term_doc_matrix.mean(axis=0)
centered = lsa_svd(centered_matrix)

pca = PCA(n_components=3).fit(term_doc_matrix)
print("PCA equals the SVD of the centered matrix:", np.allclose(pca.singular_values_, centered["sigma"][:3]))
print("Share of the sum of squares of the first three components:", centered["share"][:3].round(3))

In [ ]:
plot_scree(centered, "Centered TF-IDF matrix")

In [ ]:
top_terms(centered)

In [ ]:
plot_documents_3d(centered, "Documents in the space of the first three components (centered)")

### Interpretation (centered)

## Comparison

In [ ]:
similarity = np.abs(raw["Vt"][:4] @ centered["Vt"][:4].T)

fig, ax = plt.subplots(figsize=(4.5, 4))
image = ax.imshow(similarity, vmin=0, vmax=1, cmap="Blues")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center")
ax.set_xticks(range(4), [f"PC{j + 1}" for j in range(4)])
ax.set_yticks(range(4), [f"PC{i + 1}" for i in range(4)])
ax.set_xlabel("Centered")
ax.set_ylabel("Raw")
plt.show()

In [ ]:
rows = []
for name, result in [("raw", raw), ("centered", centered)]:
    for n_dims in (2, 3):
        coordinates = result["scores"][:, :n_dims]
        rows.append({
            "analysis": name,
            "components": n_dims,
            "silhouette": silhouette_score(coordinates, topics),
            "LDA accuracy": cross_val_score(LinearDiscriminantAnalysis(), coordinates, topics, cv=5).mean(),
        })

pd.DataFrame(rows).round(3)

### Discussion